# 08 · Phases 7–8: report + demo
1. **Build the report:** collects every finished run (metrics, comparisons, robustness, Grad-CAM
   galleries) into `MyDrive/deepfake_project/report/results.md` plus `figures/`, and zips it for download.
   Copy the contents into the repo's `reports/` folder and commit.
2. **Launch the demo:** Gradio app with a public link for the review. The cell keeps running while
   the app is live; stop it to shut the app down.

In [ ]:
# --- Session setup: run this first, every time you open the notebook ---
REPO_URL = "https://github.com/tanishq1101/robust-multimodal-deepfake.git"  # "" = use code copied to Drive

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

if REPO_URL:
    CODE = "/content/DeepFake"
    if os.path.exists(CODE):
        subprocess.run(["git", "-C", CODE, "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, CODE], check=True)
else:
    # No GitHub repo yet: upload the project folder to Google Drive at MyDrive/deepfake_project/code
    CODE = "/content/drive/MyDrive/deepfake_project/code"
    assert os.path.exists(CODE), f"Upload the project folder to {CODE} or set REPO_URL"

os.chdir(CODE)
!pip install -q -r requirements-colab.txt && pip install -q -e .
sys.path.insert(0, f"{CODE}/src")

from deepfake.config import load_config
cfg = load_config()
print("data_root (Drive):", cfg["data_root"])
print("raw data (VM disk):", cfg["paths"]["raw"])

## 1. Build the report

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
REPORT = Path(cfg["data_root"]) / "report"
# optional: write your observations once, they are included in results.md
# (REPORT / "observations.md").write_text("- Fusion fixes ...")
!python -m deepfake.report --out {REPORT}
display(Markdown((REPORT / "results.md").read_text()[:6000]))

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/report", "zip", REPORT)
files.download(zip_path)   # unzip into the repo's reports/ folder, then commit

## 2. Try one clip from the command line

In [ ]:
from deepfake.inference import find_best_checkpoint
print("best checkpoint:", find_best_checkpoint(cfg))
# upload a clip first:  from google.colab import files; files.upload()
CLIP = ""   # e.g. "/content/my_clip.mp4"
if CLIP:
    !python -m deepfake.inference "$CLIP" --out /content/clip_result

## 3. Launch the demo (public link)

In [ ]:
!python app/app.py --share